# Lab 02 — Intensity Transformations & Histogram Processing

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Power-law (Gamma) intensity correction
2. Sigmoid contrast stretching with varying slope parameter $E$
3. Histogram equalization via Look-Up Table (LUT) and Cumulative Distribution Function (CDF)
4. Histogram matching (specification)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as imageio
from skimage import exposure

IMAGES_DIR = Path("../Images")

## 1. Gamma Correction

Power-law transformation $s = c \cdot r^{\gamma}$ adjusts image brightness and mid-tone contrast non-linearly.

In [ ]:
img = imageio.imread(IMAGES_DIR / "osteoblaste.jpg")
img_norm = img.astype(np.float64) / np.max(img)

gammas = [0.5, 1.0, 2.0, 3.5]
fig, axes = plt.subplots(1, len(gammas), figsize=(16, 4))
for ax, g in zip(axes, gammas):
    adjusted = exposure.adjust_gamma(img_norm, gamma=g)
    ax.imshow(adjusted, cmap="gray")
    ax.set_title(f"Gamma = {g}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2. Sigmoid Contrast Stretching

Using the mean intensity $m$ as pivot, the sigmoid transfer function:

$$s(r) = \frac{1}{1 + \left(\frac{m}{r + \varepsilon}\right)^E}$$

steepens around $m$ as $E$ increases, approaching binary thresholding for very large $E$.

In [ ]:
def contrast_stretching(image: np.ndarray, slope: float) -> np.ndarray:
    """Apply sigmoid contrast stretching centered on the mean intensity."""
    eps = sys.float_info.epsilon
    img_float = image.astype(np.float64)
    mean_val = np.mean(img_float)
    return 1.0 / (1.0 + (mean_val / (img_float + eps)) ** slope)


slopes = [10, 20, 1000]
fig, axes = plt.subplots(1, len(slopes) + 1, figsize=(16, 4))
axes[0].imshow(img_norm, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for ax, E in zip(axes[1:], slopes):
    stretched = contrast_stretching(img_norm, E)
    ax.imshow(stretched, cmap="gray")
    ax.set_title(f"Sigmoid Stretch (E = {E})")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Histogram Equalization & CDF Analysis

We implement LUT-based histogram equalization and histogram matching, then compare the original image and equalized image along with their histograms and CDF transfer curve.

In [ ]:
def histogram_equalization(image_uint8: np.ndarray) -> np.ndarray:
    """Histogram equalization via Look-Up Table (LUT) for an 8-bit image."""
    hist, _ = np.histogram(image_uint8.ravel(), bins=256, range=(0, 256))
    cdf = hist.cumsum()
    cdf = cdf / cdf[-1]
    return cdf[image_uint8]


def histogram_matching(image: np.ndarray, target_cdf: np.ndarray) -> np.ndarray:
    """Match the histogram of `image` to a normalized target CDF."""
    imhist, bins = np.histogram(image.ravel(), bins=len(target_cdf), density=True)
    cdf = imhist.cumsum()
    cdf = cdf / cdf[-1]
    mapped = np.interp(image.ravel(), bins[:-1], cdf)
    matched = np.interp(mapped, target_cdf, bins[:-1])
    return matched.reshape(image.shape)


img_gray = img_norm.mean(axis=2) if img_norm.ndim == 3 else img_norm
img_u8 = (255 * img_gray).astype(np.uint8)
img_eq = histogram_equalization(img_u8)

hist_orig, bins = np.histogram(img_u8.ravel(), bins=256, range=(0, 256))
hist_eq, _ = np.histogram((255 * img_eq).astype(np.uint8).ravel(), bins=256, range=(0, 256))
cdf = hist_orig.cumsum() / hist_orig.sum()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].imshow(img_gray, cmap="gray")
axes[0, 0].set_title("Original Image")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_eq, cmap="gray")
axes[0, 1].set_title("Histogram Equalized (Custom LUT)")
axes[0, 1].axis("off")

axes[1, 0].bar(bins[:-1], hist_orig, width=1, color="steelblue", alpha=0.8)
axes[1, 0].set_title("Original Histogram & CDF")
axes[1, 0].set_xlabel("Intensity")
axes[1, 0].set_ylabel("Pixel Count")
ax_cdf = axes[1, 0].twinx()
ax_cdf.plot(np.arange(len(cdf)), cdf, color="crimson", linewidth=2, label="CDF (LUT)")
ax_cdf.set_ylabel("CDF", color="crimson")

axes[1, 1].bar(bins[:-1], hist_eq, width=1, color="seagreen", alpha=0.8)
axes[1, 1].set_title("Equalized Histogram")
axes[1, 1].set_xlabel("Intensity")
axes[1, 1].set_ylabel("Pixel Count")

plt.tight_layout()
plt.show()